# Collection Write Policy

**`CollectionWritePolicy`** (plugin_id `collection:write_policy`) controls what happens when an
ingested feature collides with an already-stored entity that shares the same identity.

It is driver-agnostic: PG, Elasticsearch, Iceberg, and DuckDB all read the same config
from the waterfall (`collection → catalog → platform → code default`).

## Identity keys

| Key | Description |
|---|---|
| `geoid` | Always a new UUID, always unique. The system identity key. Never duplicated — even `new_version` creates a fresh geoid for the new record. |
| `external_id` | Optional. When `external_id_field` is set, a fresh geoid is always assigned and conflict resolution uses `external_id`. When absent, conflict resolution uses geoid directly. |

## `WriteConflictPolicy` — item-level (per entity)

| Value | Behaviour |
|---|---|
| `update` | Overwrite the existing entity in place (**default**) |
| `new_version` | Archive the old version; insert the new one with a fresh geoid |
| `refuse` | Skip this entity; continue processing the rest of the batch |

## `AssetConflictPolicy` — batch-level (entire asset batch)

| Value | Behaviour |
|---|---|
| `refuse_asset` | Hard-stop — reject the entire asset batch on first duplicate |

## Composable policies

`on_conflict` (item-level) and `on_asset_conflict` (batch-level) operate independently
and can be combined.  Example — refuse individual duplicates AND abort the whole batch:

```python
{
    "on_conflict": "refuse",
    "on_asset_conflict": "refuse_asset",
}
```

## Optional fields

| Field | Default | Purpose |
|---|---|---|
| `enable_validity` | `False` | Track `valid_from` / `valid_to` per entity |
| `validity_field` | `valid_from` | Source field to extract validity start from |
| `track_asset_id` | `True` | Stamp each entity with the ingestion `asset_id` |
| `external_id_field` | `None` | Dot-path to extract external_id (e.g. `"id"`, `"properties.code"`). When `None`, conflict uses geoid. |
| `require_external_id` | `False` | Reject entities that lack an external_id |

Drivers that declare `Capability.TEMPORAL_VALIDITY` honour `enable_validity`.

In [ ]:
import httpx
import asyncio
import os
BASE = os.environ.get("DYNASTORE_BASE_URL") or "http://localhost:8080"
CATALOG_ID = "demo-write-policy"

client = httpx.AsyncClient(base_url=BASE, timeout=30)

def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    suffix = "" if ok else f" — {r.text[:300]}"
    print(f"{label + ': ' if label else ''}{status}{suffix}")
    return r

# Create catalog once
r = await client.post("/stac/catalogs", json={"id": CATALOG_ID, "title": "Write Policy Demo", "description": "Write Policy Demo catalog."})
_check(r, "Catalog")

# Wait for provisioning to complete before creating collections
for _ in range(30):
    r = await client.get(f"/stac/catalogs/{CATALOG_ID}")
    if r.status_code == 200 and r.json().get("provisioning_status") == "ready":
        print("Catalog ready")
        break
    await asyncio.sleep(1)
else:
    print("Warning: catalog still provisioning after 30s")

---
## 1. `update` — overwrite in place (default)

Re-ingesting the same feature replaces the stored record.  Use when the source
of truth is always the latest snapshot (e.g. sensor readings, daily reports).

In [ ]:
COLL_UPDATE = "sensors-update"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_UPDATE,
    "title": "Sensors — UPDATE policy",
    "description": "Sensor readings with UPDATE write policy — re-ingest overwrites in place.",
    "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r, "Collection")

# Set write policy — external_id_field defaults to None (use geoid for conflict resolution)
policy = {"on_conflict": "update", "track_asset_id": True, "external_id_field": "id"}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_UPDATE}/classes/CollectionWritePolicy",
    json=policy,
)
_check(r, "Policy")

In [ ]:
feature = {
    "type": "Feature",
    "id": "sensor-001",
    "geometry": {"type": "Point", "coordinates": [12.49, 41.89]},
    "bbox": [12.49, 41.89, 12.49, 41.89],
    "properties": {"temperature": 22.3, "status": "nominal"},
}

# First ingest
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_UPDATE}/items",
    json=feature,
)
_check(r, "First ingest")

# Re-ingest with updated temperature — overwrites
feature["properties"]["temperature"] = 25.1
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_UPDATE}/items",
    json=feature,
)
_check(r, "Re-ingest (update)")

r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_UPDATE}/items?limit=10")
items = r.json().get("features", [])
if items:
    print("Stored temperature:", items[0]["properties"]["temperature"])  # 25.1
else:
    print("No items stored")

---
## 2. `new_version` — temporal versioning

Each re-ingest creates a new temporal row; the previous version is archived.
Use for datasets where history matters (regulations, auditing, change detection).

Enable `enable_validity` to track the `valid_from` / `valid_to` window
extracted from the feature payload.

In [ ]:
COLL_VERSION = "land-use-versioned"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_VERSION,
    "title": "Land Use — NEW_VERSION policy",
    "description": "Land use parcels with temporal versioning — each re-ingest archives the previous version.",
    "license": "CC-BY-4.0",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r, "Collection")

policy = {
    "on_conflict": "new_version",
    "enable_validity": True,
    "validity_field": "reference_year",
    "external_id_field": "properties.parcel_id",
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_VERSION}/classes/CollectionWritePolicy",
    json=policy,
)
_check(r, "Policy")

In [ ]:
parcel = {
    "type": "Feature",
    "id": "P-00123",
    "geometry": {"type": "Polygon", "coordinates": [[[12.0, 41.0], [12.1, 41.0], [12.1, 41.1], [12.0, 41.1], [12.0, 41.0]]]},
    "bbox": [12.0, 41.0, 12.1, 41.1],
    "properties": {"parcel_id": "P-00123", "land_class": "arable", "reference_year": "2022"},
}

# 2022 version
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_VERSION}/items", json=parcel,
)
_check(r, "2022 ingest")

# 2024 reclassification — creates new version, archives 2022
parcel["properties"]["land_class"] = "forest"
parcel["properties"]["reference_year"] = "2024"
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_VERSION}/items", json=parcel,
)
_check(r, "2024 ingest (new version)")

# Browse all versions
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_VERSION}/items?limit=10")
items = r.json().get("features", [])
print(f"Stored versions: {len(items)}")
for it in items:
    p = it["properties"]
    print(f"  parcel={p.get('parcel_id')} class={p.get('land_class')} year={p.get('reference_year')}")

---
## 3. `refuse` — skip duplicate, continue batch

If the identity already exists, the incoming entity is silently discarded.
The rest of the batch continues normally.  Use for idempotent pipelines where
duplicate delivery must not overwrite authoritative records.

In [ ]:
COLL_REFUSE = "boundaries-refuse"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_REFUSE,
    "title": "Boundaries — REFUSE policy",
    "description": "Administrative boundaries with REFUSE write policy — duplicates are silently skipped.",
    "license": "ODbL",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r, "Collection")

policy = {"on_conflict": "refuse", "external_id_field": "properties.iso3"}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/classes/CollectionWritePolicy",
    json=policy,
)
_check(r, "Policy")

batch = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "id": "ITA", "geometry": {"type": "Point", "coordinates": [12.5, 41.9]},
     "bbox": [12.5, 41.9, 12.5, 41.9],
     "properties": {"iso3": "ITA", "name": "Italy"}},
    {"type": "Feature", "id": "FRA", "geometry": {"type": "Point", "coordinates": [2.3, 48.9]},
     "bbox": [2.3, 48.9, 2.3, 48.9],
     "properties": {"iso3": "FRA", "name": "France"}},
]}

r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/items", json=batch,
)
_check(r, "First batch")

# Attempt to re-ingest Italy + add Germany — Italy is refused, Germany written
batch2 = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "id": "ITA", "geometry": {"type": "Point", "coordinates": [12.5, 41.9]},
     "bbox": [12.5, 41.9, 12.5, 41.9],
     "properties": {"iso3": "ITA", "name": "Italy MODIFIED"}},  # duplicate → refused
    {"type": "Feature", "id": "DEU", "geometry": {"type": "Point", "coordinates": [10.4, 51.2]},
     "bbox": [10.4, 51.2, 10.4, 51.2],
     "properties": {"iso3": "DEU", "name": "Germany"}},          # new → written
]}
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/items", json=batch2,
)
_check(r, "Second batch (Italy refused, Germany written)")

r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/items")
items = r.json().get("features", [])
print(f"Total stored: {len(items)} (ITA unchanged, DEU added)")
for it in items:
    print("  ", it["properties"]["iso3"], "-", it["properties"]["name"])

---
## 4. `refuse_asset` — hard-stop on duplicate (batch-level)

Set `on_asset_conflict = "refuse_asset"` to abort the **entire asset batch** the moment
any entity's external_id already exists in the collection.  Use for high-integrity
pipelines (financial records, legal boundaries) where accidental duplicates must never
go unnoticed.

This is the batch-level policy.  It operates independently from `on_conflict` (item-level)
and both can be combined.

In [ ]:
COLL_HARD = "parcels-strict"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_HARD,
    "title": "Parcels — REFUSE_ASSET policy",
    "description": "Cadastral parcels with hard-stop batch policy — entire batch rejected on first duplicate.",
    "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r, "Collection")

policy = {
    "on_asset_conflict": "refuse_asset",  # batch-level: abort entire batch on first duplicate
    "require_external_id": True,
    "external_id_field": "properties.cadastral_id",
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_HARD}/classes/CollectionWritePolicy",
    json=policy,
)
_check(r, "Policy")

parcel = {"type": "Feature", "id": "CAD-9912",
          "geometry": {"type": "Point", "coordinates": [11.2, 43.8]},
          "bbox": [11.2, 43.8, 11.2, 43.8],
          "properties": {"cadastral_id": "CAD-9912", "area_m2": 5200}}

r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_HARD}/items", json=parcel,
)
_check(r, "First ingest")

# Duplicate → entire batch rejected with 409
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_HARD}/items", json=parcel,
)
_check(r, "Duplicate ingest — expected 409")
if r.status_code == 409:
    print("  ✓ Batch correctly rejected:", r.json().get("detail", r.text[:100]))

---
## Cascade: collection-level policy overrides catalog default

The config waterfall means you can set a platform/catalog default and override
at collection level for specific datasets.

In [ ]:
# Set catalog-level default: refuse (protect all collections by default)
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/classes/CollectionWritePolicy",
    json={"on_conflict": "refuse"},
)
_check(r, "Catalog default (refuse)")

# Override one collection to allow updates
COLL_OVERRIDE = "snapshots-update"
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_OVERRIDE,
    "title": "Snapshots (update override)",
    "description": "Snapshot collection overriding catalog default — allows UPDATE despite catalog-level REFUSE.",
    "license": "CC-BY-4.0",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
    "collection:write_policy": {"on_conflict": "update"},  # embedded at creation
})
_check(r, "Collection with embedded write_policy")

In [ ]:
# Cleanup
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
_check(r, "Cleanup")
await client.aclose()